In [2]:
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


# Setting

In [3]:
from dotenv import load_dotenv
import os

load_dotenv()

user = os.getenv("DB_USER")
password = os.getenv("DB_PASS")
host = os.getenv("DB_HOST")
dbname = os.getenv("DB_NAME_SCHOOL")

from sqlalchemy import create_engine
engine = create_engine(f"mysql+mysqlconnector://{user}:{password}@{host}/{dbname}", connect_args={'init_command': 'SET time_zone="+07:00"'})

# Data

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sqlalchemy import create_engine, text

In [5]:
query = """select MAMH, DIEM_HE4, TINCHI
        from  MONHOC natural join KETQUA
        where  TENMH not like '%Giáo dục thể chất%' and TENMH not like '%Bóng đá%'
        """
df = pd.read_sql(query, engine)

In [6]:
df.head()

,MAMH,DIEM_HE4,TINCHI
0,801043,2.0,4
1,801046,1.0,4
2,801047,1.0,4
3,801145,2.0,3
4,801351,4.0,2


In [7]:
df1 = pd.DataFrame({'MAMH': ["841047", # Công nghệ phần mền
                             "848028", # Phân tích xử lý ảnh
                             "tanh123", # Tiếng Anh 123
                             ], 
                    'DIEM_HE4': [4.0,
                                 4.0,
                                 4.0,
                                 ], 
                    'TINCHI': [4,
                               4,
                               10,
                               ]})
# df1 = pd.DataFrame()
df1.head()

,MAMH,DIEM_HE4,TINCHI
0,841047,4.0,4
1,848028,4.0,4
2,tanh123,4.0,10


In [8]:
union_all = pd.concat([df, df1], ignore_index=True)

In [9]:
from sqlalchemy import text

def update_caithien(mamh, diem):
    with engine.begin() as conn:  # begin() auto-commits or rolls back
        conn.execute(
            text("""
                UPDATE `KETQUA`
                SET `DIEM_HE4` = :diem
                WHERE `MAMH` = ":mamh"
            """),
            {"diem": diem, "mamh": mamh}
        )

In [10]:
# update_caithien(801401, 0.0) # diem goc mon dstt
# update_caithien(841403, 1.0) # diem goc mon ctrr
update_caithien(801401, 4.0) # diem cai thien mon dstt
update_caithien(841403, 4.0) # diem cai thien mon ctrr

In [11]:
gpa = (union_all['DIEM_HE4'] * union_all['TINCHI']).sum() / union_all['TINCHI'].sum()
print(f"GPA: {gpa:.2f}")

GPA: 3.02
